In [12]:
# config.py
import os
from dotenv import load_dotenv

# Optional: load from .env file if you're using one
load_dotenv()

# ---- API KEYS ----
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

# ---- PINECONE SETTINGS ----
PINECONE_INDEX_NAME = "hr-policy-gesci-index"  # choose any name (must exist or will be created)

# ---- MODEL NAMES ----
# Choose appropriate models (update as per your access)
EMBEDDING_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4.1-mini"  # or "gpt-4o-mini"
PDF_PATH = "HR_policy_gesci.pdf"


In [13]:
import time
from typing import List

# UPDATED: Loaders remain in community, but best practice is to be explicit
from langchain_community.document_loaders import PyPDFLoader

# UPDATED: Text Splitters now live in their own package 'langchain_text_splitters'
from langchain_text_splitters import RecursiveCharacterTextSplitter

# UPDATED: Core definitions for type hinting
from langchain_core.documents import Document

# Partner packages (Correct as per v0.1+)
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

# Native Pinecone client for Admin tasks (Index creation)
from pinecone import Pinecone, ServerlessSpec


In [14]:
def load_and_split_pdf(pdf_path):
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
    )
    return splitter.split_documents(docs)

In [15]:
def init_pinecone():
    pc = Pinecone(api_key=PINECONE_API_KEY)

    existing_indexes = [idx["name"] for idx in pc.list_indexes()]

    if PINECONE_INDEX_NAME not in existing_indexes:
        pc.create_index(
            name=PINECONE_INDEX_NAME,
            dimension=1536,
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1",
            ),
        )
        print("Pinecone index created")
    else:
        print("Pinecone index already exists")

    return pc


In [16]:

def index_pdf():
    if not os.path.exists(PDF_PATH):
        raise FileNotFoundError(f"{PDF_PATH} not found")

    print("Loading and splitting PDF...")
    docs = load_and_split_pdf(PDF_PATH)

    print(f"Total chunks: {len(docs)}")

    embeddings = OpenAIEmbeddings(
        openai_api_key=OPENAI_API_KEY,
        model=EMBEDDING_MODEL,
    )

    init_pinecone()

    PineconeVectorStore.from_documents(
        documents=docs,
        embedding=embeddings,
        index_name=PINECONE_INDEX_NAME,
    )

    print("PDF successfully indexed into Pinecone")

In [17]:

if __name__ == "__main__":
    index_pdf()

Loading and splitting PDF...
Total chunks: 247
Pinecone index created
PDF successfully indexed into Pinecone


In [18]:
from langchain_openai import ChatOpenAI

def get_retriever():
    """Connect to Pinecone vector store and return a retriever."""
    embeddings = OpenAIEmbeddings(
        openai_api_key=OPENAI_API_KEY,
        model=EMBEDDING_MODEL,
    )

    vectorstore = PineconeVectorStore(
        index_name=PINECONE_INDEX_NAME,
        embedding=embeddings,
    )

    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
    return retriever

In [19]:
def build_llm():
    """Return the chat model."""
    llm = ChatOpenAI(
        openai_api_key=OPENAI_API_KEY,
        model=CHAT_MODEL,
        temperature=0.1,  # low temp for factual answers
    )
    return llm

In [20]:
def answer_question(query: str, retriever, llm):
    """RAG-style answering: retrieve docs, then ask LLM using those docs as context."""
    docs = retriever.invoke(query)

    # Build context string from retrieved chunks
    context_parts = []
    for doc in docs:
        page = doc.metadata.get("page", "N/A")
        context_parts.append(f"(Page {page}) {doc.page_content}")
    context = "\n\n".join(context_parts)

    prompt = f"""
You are an HR assistant chatbot. Answer the user's question *only* using the context from the HR policy below.
If the answer is not clearly present, say you are not sure and ask the user to check with HR.

Context from HR Policy:
-----------------------
{context}
-----------------------

Question: {query}

Answer in a clear and concise way:
"""

    response = llm.invoke(prompt)
    # response is usually an object with `.content`
    return getattr(response, "content", str(response))

In [21]:
import textwrap
def chat():
    print("HR Policy RAG Chatbot Ready (type 'exit' to quit)")
    retriever = get_retriever()
    llm = build_llm()

    while True:
        query = input("\nYour Question: ").strip()
        if query.lower() in {"exit", "quit"}:
            print("Goodbye!")
            break

        try:
            answer = answer_question(query, retriever, llm)
            print("\n--- Answer ---")
            print(textwrap.fill(answer.strip(),width=100))
        except Exception as e:
            print("\nError while answering:", e)

In [22]:
if __name__ == "__main__":
    chat()

HR Policy RAG Chatbot Ready (type 'exit' to quit)
Goodbye!
